In [ ]:
!pip install steamreviews
import steamreviews as sr


The code below fetches steam reviews

Documentation:
https://pypi.org/project/steamreviews/

https://partner.steamgames.com/doc/store/getreviews

In [ ]:
# List of games

# 40 = Deathmatch Classic
# 570 = Dota 2
# 440 = TF2
# 500 = Left 4 Dead
# 550 = Left 4 Dead 2
# 240 = Counter-Strike Source
# 730 = Counter-Strike 2
# 282800 = 100% Orange Juice
# 394690 = Tower Unite
# 17500 = Zombie Panic! Source
app_ids = [40, 570, 440, 500, 550, 240, 730, 282800, 394690, 17500]

In [ ]:
# Review Filters
# Reference: https://partner.steamgames.com/doc/store/getreviews

# Request parameters
request_params = {
    'language': 'english',
    'purchase_type': 'steam',
    'day_range': 365
}

In [ ]:
# Get reviews
review_dict, query_count = sr.download_reviews_for_app_id_batch(app_ids, chosen_request_params=request_params)

# **Review Filtering**

Imports and Setup

In [ ]:
!pip install steamreviews

import os
import json
import re
import csv
import nltk
from nltk.sentiment.vader import SentimentIntensityAnalyzer

# Download the VADER lexicon (only needed once)
nltk.download('vader_lexicon')


Configuration

In [ ]:
# Directory with review_XXX.json files
data_folder = 'data'

# Keywords that relate to cheating/VAC
keywords = [
    'vac', 'cheat', 'cheater', 'cheating', 'hack', 'hacker',
    'aimbot', 'wallhack', 'ban', 'banned', 'script', 'scripting', 'rage', 'spinbot'
]
keyword_pattern = re.compile(r'\b(' + '|'.join(keywords) + r')\b', re.IGNORECASE)

# Initialize VADER sentiment analyzer
analyzer = SentimentIntensityAnalyzer()


Process Files

In [ ]:
# Loop through review files in the /data directory
for filename in os.listdir(data_folder):
    if filename.endswith(".json") and filename.startswith("review_"):
        app_id = filename.replace("review_", "").replace(".json", "")
        filepath = os.path.join(data_folder, filename)

        # Load review JSON
        with open(filepath, 'r', encoding='utf-8') as f:
            data = json.load(f)

        filtered_reviews = []

        for review_id, review in data.get("reviews", {}).items():
            text = review.get("review", "")
            if keyword_pattern.search(text):
                sentiment = analyzer.polarity_scores(text)
                compound = sentiment['compound']
                label = 'positive' if compound > 0.05 else 'negative' if compound < -0.05 else 'neutral'

                filtered_reviews.append({
                    'review_id': review_id,
                    'app_id': app_id,
                    'text': text,
                    'voted_up': review.get('voted_up'),
                    'playtime_minutes': review['author'].get('playtime_forever', 0),
                    'timestamp': review.get('timestamp_created'),
                    'sentiment_score': compound,
                    'label': label
                })


Save Results as CSV

In [ ]:

        # Save filtered results to CSV
        if filtered_reviews:
            output_path = os.path.join(data_folder, f"filtered_{app_id}.csv")
            with open(output_path, 'w', encoding='utf-8', newline='') as csvfile:
                writer = csv.DictWriter(csvfile, fieldnames=filtered_reviews[0].keys())
                writer.writeheader()
                writer.writerows(filtered_reviews)
            print(f"✅ Saved {len(filtered_reviews)} reviews to {output_path}")
        else:
            print(f"⚠️ No cheating-related reviews found in {filename}")